In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gemini-2.5-flash-lite'

In [5]:
# Gemini OpenAI-compatible endpoint
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# Load .env file
load_dotenv(override=True)

# Read API key
google_api_key = os.getenv("GOOGLE_API_KEY")

# Validate key
if not google_api_key:
    print("No API key was found - please add your key to the .env file")
elif not google_api_key.startswith("AIza"):
    print("An API key was found, but it doesn't start with AIza")
else:
    print("API key found and looks good so far!")

# Connect to Gemini
gemini = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=google_api_key
)

# Ask Gemini
response = gemini.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {
            "role": "user",
            "content": "Hi"
        }
    ]
)

# Print response
print(response.choices[0].message.content)

API key found and looks good so far!
Hi there! How can I help you today?


10 deals from each feed.

So:

Electronics → 10 deals

Computers   → 10 deals

Smart Home  → 10 deals

↓

~30 deals total

In [6]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:49<00:00, 16.56s/it]


In [7]:
len(deals)

30

In [8]:
deals[17].describe()

"Title: Satechi 5-in-1 USB Type-C Combo Hub for $27 + free shipping\nDetails: Adorama offers the Satechi 5-in-1 USB Type-C Combo Hub for $26.99.  That's an $8 low.  Shipping is free. Buy Now at Adorama\nFeatures: \nURL: https://www.dealnews.com/products/Satechi/5-in-1-USB-Type-C-Combo-Hub/499166.html?iref=rss-c39"

In [9]:
deals[29].describe()

"Title: Philips Hue Bridge Pro Smart Lighting Hub for $99 + free shipping\nDetails: At Amazon, get the Philips Hue Bridge Pro Smart Lighting Hub for $99. It's the best price we could find by $41.The Philips Hue Bridge Pro supports up to 150 lights and 50 accessories, with 8 GB of onboard memory for storing up to 500 scenes and automations. It runs on a 1.7 GHz quad-core processor and uses Zigbee Trust Center encryption for local data protection. Buy Now at Amazon\nFeatures: 1.7 GHz quad-core processor (Hue Chip Pro) Supports up to 150 lights & 50 accessories 8 GB memory, up to 500 scenes & automations Zigbee Trust Center advanced encryption Compatible with Apple Home, Alexa, Google, Samsung SmartThings Wireless connectivity\nURL: https://www.dealnews.com/Philips-Hue-Bridge-Pro-Smart-Lighting-Hub-for-99-free-shipping/21839519.html?iref=rss-f1912"

## We are going to ask GEMINI to summarize deals and identify their price

In [10]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [11]:
def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [12]:
user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: NUU B30 5G Unlocked Android Smartphones for 2 for $279.99 + free shipping
Details: Budget-conscious users who want a 5G-capable phone without committing to flagship pricing would find the NUU B30 a reasonable option, get two NUU B30 5G Unlocked Android Smartphones for the price of one. That's a savings of $280. Buy Now at Nuu Mobile
Features: 6.7-inch AMOLED display, 1080x2400 

In [13]:
response = gemini.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='This professional video support kit includes the Manfrotto 502AH fluid video head, designed for smooth panning and tilting with a 22-lb payload capacity. It is paired with the MT055XPRO3 aluminum tripod, which extends to a maximum height of 67 inches. The kit features a sliding quick-release plate for easy camera mounting and comes with an included pan bar for precise control.', price=299.0, url='https://www.dealnews.com/Manfrotto-502-AH-Video-Head-Tripod-Kit-for-299-free-shipping/21840064.html?iref=rss-c142'), Deal(product_description='The Dreame L40s Ultra CE is a robot vacuum and mop combination unit designed for high-traffic areas and homes with pets. It boasts 13,000Pa suction power for deep cleaning floors and carpets, and can go hands-free for up to 100 days with its self-emptying dock. The device automatically washes, refills, and hot-air dries its mop pads, and features a TriCut anti-tangle brush to manage hair effectively.', pric

In [14]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    

This professional video support kit includes the Manfrotto 502AH fluid video head, designed for smooth panning and tilting with a 22-lb payload capacity. It is paired with the MT055XPRO3 aluminum tripod, which extends to a maximum height of 67 inches. The kit features a sliding quick-release plate for easy camera mounting and comes with an included pan bar for precise control.
299.0
https://www.dealnews.com/Manfrotto-502-AH-Video-Head-Tripod-Kit-for-299-free-shipping/21840064.html?iref=rss-c142
The Dreame L40s Ultra CE is a robot vacuum and mop combination unit designed for high-traffic areas and homes with pets. It boasts 13,000Pa suction power for deep cleaning floors and carpets, and can go hands-free for up to 100 days with its self-emptying dock. The device automatically washes, refills, and hot-air dries its mop pads, and features a TriCut anti-tangle brush to manage hair effectively.
399.0
https://www.dealnews.com/Dreame-L40-s-Ultra-CE-Robot-Vacuum-and-Mop-for-399-free-shipping/

In [15]:
root = logging.getLogger()
root.setLevel(logging.INFO)

## Scanning Agent

In [16]:
from agents.scanner_agent import ScannerAgent

In [ ]:
agent = ScannerAgent()
result = agent.scan()

In [ ]:
result

## Scanner Agent workflow

Scanner Agent finds a deal:

Product: Sony Microphone

Deal Price = $300

Now Scanner Agent asks the Ensemble Agent:

"What should this product actually cost?"

The Ensemble Agent calculates:

Frontier Agent   = $420

Specialist Agent = $380

Neural Network   = $400

↓

Weighted average:

Ensemble Price ≈ $400

Now Scanner Agent compares:

Deal Price      = $300

Ensemble Price  = $400

Difference:

Potential Saving = $100

Since:

300 < 400

the product is considered a good deal/opportunity.

# Introducing Pushover

Pushover is a service that sends notifications directly to your phone.



In [17]:
load_dotenv(override=True)

True

In [18]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [19]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [20]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [21]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [22]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [23]:
agent.notify(
    "Wrogn Brown Formal Shirt available at a great discount",
    799,
    1499,
    "https://www.wrogn.com"
)

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
15:41:28 - LiteLLM:INFO: utils.py:4083 - 
LiteLLM completion() model= llama3.2; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= llama3.2; provider = ollama
15:41:45 - LiteLLM:INFO: utils.py:1656 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed


## Messaging Agent workflow

Deal Price = $300

Estimated Value = $450

↓

Good Opportunity

↓

Messaging Agent

↓

Notification Sent